In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle


In [ ]:
### Load the dataset
data = pd.read_csv('Churn_Modelling.csv')
data.head()

In [ ]:
## preprocess the data
### Drop irrelevant columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data


In [ ]:
### encode categorical variables
label_encode_gender=LabelEncoder()
data['Gender']=label_encode_gender.fit_transform(data['Gender'])
data


In [ ]:
## one-hot encode the 'Geography' column
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoded

In [ ]:
geo_encoded.toarray()

In [ ]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

In [ ]:
geo_encoded_df = pd.DataFrame(geo_encoded.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography'])) 
geo_encoded_df   

In [ ]:
## combine the one-hot encoded columns with the original dataframe
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

In [ ]:
## save the encoder and scaler
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encode_gender, file)
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)
    

In [ ]:
data.head()

In [ ]:
## Divde the dataset into independent and dependent features
x = data.drop('Exited', axis=1)
y = data['Exited']

## Split the data in taining and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## scale the features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


In [ ]:
x_train


In [ ]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [ ]:
data

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [ ]:
(x_train.shape[1],)

In [ ]:
#Build our ANN model
model = Sequential([
    Dense(64, activation='relu', input_shape=(x_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
]

)


In [ ]:
model.summary()

In [ ]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.001)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

In [ ]:
## Compile the model
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=['accuracy'])


In [ ]:
## setup the tensorboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)



In [ ]:
##set up early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)  

In [ ]:
## train the model
history=model.fit(
    x_train, y_train, validation_data=(x_test, y_test), epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
from tensorflow.keras.callbacks import TensorBoard

tensorboard_callback = TensorBoard(log_dir="logs/fit")

model.fit(x_train, y_train, epochs=10, callbacks=[tensorboard_callback])

In [ ]:
from tensorflow.keras.callbacks import TensorBoard

tensorboard_callback = TensorBoard(log_dir="logs/fit")

model.fit(x_train, y_train, epochs=10, callbacks=[tensorboard_callback])

In [ ]:
## load tensorboard extension
%reload_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit20260409-132655

In [ ]:
## load the picle files
